# EDA — International Football Results 1872–2026
**Objetivo:** entender la distribución del dataset antes de construir el modelo.

Dataset: Kaggle — `martj42/international-football-results-from-1872-to-2017`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.extractor import load_results, filter_world_cups, add_outcome

## 1. Dataset completo

In [ ]:
df_all = load_results()
print(f"Shape: {df_all.shape}")
print(f"Rango: {df_all['date'].min().date()} → {df_all['date'].max().date()}")
print(f"Equipos únicos: {df_all['home_team'].nunique()}")
print(f"Torneos únicos: {df_all['tournament'].nunique()}")
df_all.head(3)

In [ ]:
# Torneos más frecuentes
top_tournaments = df_all['tournament'].value_counts().head(12).reset_index()
top_tournaments.columns = ['tournament', 'count']

fig = px.bar(top_tournaments, x='count', y='tournament', orientation='h',
             title='Top 12 torneos por número de partidos',
             color='count', color_continuous_scale='Reds')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, showlegend=False)
fig.show()

## 2. Filtrado — solo FIFA World Cup (fase final)

In [ ]:
df_wc = filter_world_cups(df_all)
df_wc = df_wc.dropna(subset=['home_score', 'away_score']).copy()
df_wc = add_outcome(df_wc)
df_wc['year'] = df_wc['date'].dt.year

print(f"Partidos de Mundial: {len(df_wc)}")
print(f"Ediciones: {df_wc['year'].nunique()} ({df_wc['year'].min()}–{df_wc['year'].max()})")
print(f"Nulos descartados (sin resultado): {len(filter_world_cups(df_all)) - len(df_wc)}")
df_wc.head(3)

## 3. Distribución de resultados (variable objetivo)

In [ ]:
outcome_counts = df_wc['outcome'].value_counts().reset_index()
outcome_counts.columns = ['outcome', 'count']
outcome_counts['pct'] = (outcome_counts['count'] / len(df_wc) * 100).round(1)

colors = {'home_win': '#C8102E', 'draw': '#888888', 'away_win': '#003087'}

fig = px.pie(
    outcome_counts, values='count', names='outcome',
    title='Distribución de resultados en Mundiales (1930–2022)',
    color='outcome', color_discrete_map=colors,
    hole=0.45
)
fig.update_traces(textinfo='label+percent')
fig.show()

print(outcome_counts.to_string(index=False))

**Hallazgo:** El local gana en el 45.5% de los partidos. El empate es la clase minoritaria (22.2%). 
Hay desequilibrio de clases — relevante para la calibración del modelo.

## 4. Evolución histórica de resultados por edición

In [ ]:
by_year = df_wc.groupby(['year', 'outcome']).size().unstack(fill_value=0)
by_year_pct = by_year.div(by_year.sum(axis=1), axis=0) * 100

fig = go.Figure()
for outcome, color in colors.items():
    if outcome in by_year_pct.columns:
        fig.add_trace(go.Scatter(
            x=by_year_pct.index, y=by_year_pct[outcome],
            name=outcome, line=dict(color=color, width=2),
            mode='lines+markers'
        ))

fig.update_layout(
    title='% de resultados por edición del Mundial',
    xaxis_title='Año', yaxis_title='% partidos',
    hovermode='x unified'
)
fig.show()

## 5. Estadísticas de goles

In [ ]:
df_wc['total_goals'] = df_wc['home_score'] + df_wc['away_score']

print("=== Goles por partido ===")
print(f"Promedio total:      {df_wc['total_goals'].mean():.2f}")
print(f"Goles local (media): {df_wc['home_score'].mean():.2f}")
print(f"Goles visit (media): {df_wc['away_score'].mean():.2f}")
print(f"Máximo goles:        {int(df_wc['total_goals'].max())} ({df_wc.loc[df_wc['total_goals'].idxmax(), ['home_team','away_team','home_score','away_score','year']].to_dict()})")

fig = px.histogram(
    df_wc, x='total_goals', nbins=15,
    title='Distribución de goles por partido (Mundiales)',
    color_discrete_sequence=['#C8102E']
)
fig.update_layout(bargap=0.1, xaxis_title='Goles totales', yaxis_title='Partidos')
fig.show()

In [ ]:
# Goles promedio por edición — tendencia histórica
goals_by_year = df_wc.groupby('year')['total_goals'].mean().reset_index()
goals_by_year.columns = ['year', 'avg_goals']

fig = px.line(goals_by_year, x='year', y='avg_goals',
              title='Goles promedio por partido en cada Mundial',
              markers=True, color_discrete_sequence=['#C8102E'])
fig.update_layout(yaxis_title='Goles promedio')
fig.show()

## 6. Selecciones más exitosas

In [ ]:
# Victorias totales (desde perspectiva del ganador)
home_wins = df_wc[df_wc['outcome'] == 'home_win']['home_team'].value_counts()
away_wins = df_wc[df_wc['outcome'] == 'away_win']['away_team'].value_counts()
total_wins = home_wins.add(away_wins, fill_value=0).astype(int).sort_values(ascending=False)

total_games = (df_wc['home_team'].value_counts().add(
    df_wc['away_team'].value_counts(), fill_value=0)).astype(int)

df_teams = pd.DataFrame({
    'wins': total_wins,
    'games': total_games,
}).dropna().astype(int)
df_teams['win_rate'] = (df_teams['wins'] / df_teams['games'] * 100).round(1)
df_teams = df_teams[df_teams['games'] >= 10].sort_values('wins', ascending=False).head(15)

fig = px.bar(df_teams.reset_index(), x='index', y='wins',
             hover_data=['games', 'win_rate'],
             title='Top 15 selecciones por victorias en Mundiales (mínimo 10 partidos)',
             color='win_rate', color_continuous_scale='RdBu')
fig.update_layout(xaxis_title='Selección', yaxis_title='Victorias')
fig.show()

df_teams.head(15)

## 7. Colombia en Mundiales

In [ ]:
col = df_wc[(df_wc['home_team'] == 'Colombia') | (df_wc['away_team'] == 'Colombia')].copy()

def colombia_result(row):
    if row['home_team'] == 'Colombia':
        return row['outcome']
    inv = {'home_win': 'away_win', 'away_win': 'home_win', 'draw': 'draw'}
    return inv[row['outcome']]

col['col_result'] = col.apply(colombia_result, axis=1)
col['opponent'] = np.where(col['home_team'] == 'Colombia', col['away_team'], col['home_team'])
col['col_goals'] = np.where(col['home_team'] == 'Colombia', col['home_score'], col['away_score'])
col['opp_goals'] = np.where(col['home_team'] == 'Colombia', col['away_score'], col['home_score'])

print(f"Partidos de Colombia en Mundiales: {len(col)}")
print(col['col_result'].value_counts().to_string())
print(f"\nGoles a favor:  {col['col_goals'].sum():.0f} ({col['col_goals'].mean():.2f}/partido)")
print(f"Goles en contra: {col['opp_goals'].sum():.0f} ({col['opp_goals'].mean():.2f}/partido)")

col[['year','opponent','col_goals','opp_goals','col_result']].sort_values('year')

## 8. Datos 2026 ya disponibles en el dataset

In [ ]:
# El dataset incluye partidos del Mundial 2026 (hasta 2026-06-27)
# Estos NO entran en entrenamiento pero son útiles para validación real
df_2026_raw = df_all[
    (df_all['tournament'] == 'FIFA World Cup') &
    (df_all['date'].dt.year == 2026)
].copy()

played = df_2026_raw.dropna(subset=['home_score', 'away_score'])
scheduled = df_2026_raw[df_2026_raw['home_score'].isna()]

print(f"Partidos 2026 en el dataset: {len(df_2026_raw)}")
print(f"  Jugados: {len(played)}")
print(f"  Sin resultado aún: {len(scheduled)}")

if len(played) > 0:
    print("\nPartidos jugados hasta ahora:")
    print(played[['date','home_team','away_team','home_score','away_score']].to_string(index=False))

## 9. Resumen de hallazgos

| Hallazgo | Valor | Implicación para el modelo |
|---|---|---|
| Partidos de Mundial | 964 | Dataset pequeño → XGBoost justificado sobre redes neuronales |
| Distribución outcome | 45.5% / 22.2% / 32.3% | Clases desbalanceadas → usar log-loss como métrica principal |
| Goles promedio | 2.82/partido | Señal de rendimiento ofensivo relevante como feature |
| Ventaja de local | +0.32 goles | `is_neutral` será una feature relevante |
| Nulos en scores | 72 (2026 aún en juego) | Excluir de entrenamiento |
| Colombia en WC | 9V 3E 10D en 22 partidos | Historial desfavorable pero con tendencia positiva reciente |